In [1]:
from collections import Counter
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.manual_seed(42)
torch.set_num_threads(2)

stream_train = load_dataset("coastalcph/lex_glue", "ledgar", split="train", streaming=True)
names = stream_train.features["label"].names

sample_rows = list(stream_train.take(4000))
TOP = [l for l, _ in Counter(r["label"] for r in sample_rows).most_common(8)]
top_set = set(TOP)

tok = AutoTokenizer.from_pretrained("distilgpt2")
tok.pad_token = tok.eos_token
tok.add_tokens([f"<{names[i]}>" for i in TOP])

label_to_token_id = {i: tok.convert_tokens_to_ids(f"<{names[i]}>") for i in TOP}

def prepare_rows(rows):
    formatted = []
    for r in rows:
        clause = " ".join(r["text"].strip().split())[:220] + " "
        enc = tok(clause, truncation=True, max_length=48)
        formatted.append({
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "label_token_id": label_to_token_id[r["label"]],
            "label_idx": r["label"],
            "text": r["text"],
        })
    return formatted

train_raw = [r for r in sample_rows if r["label"] in top_set][:480]
train_ds = Dataset.from_list(prepare_rows(train_raw))

stream_val = load_dataset("coastalcph/lex_glue", "ledgar", split="validation", streaming=True)
val_raw = [r for r in stream_val.take(2000) if r["label"] in top_set][:120]

model = AutoModelForCausalLM.from_pretrained("distilgpt2", low_cpu_mem_usage=True)
model.resize_token_embeddings(len(tok))

print(f"Clause types ({len(TOP)}): {[names[i] for i in TOP]}")
print(f"Train samples: {len(train_ds)} | Held-out val samples: {len(val_raw)}")

/opt/conda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 76/76 [00:02<00:00, 31.03it/s] 
[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Clause types (8): ['Governing Laws', 'Counterparts', 'Notices', 'Entire Agreements', 'Severability', 'Survival', 'Amendments', 'Assignments']
Train samples: 480 | Held-out val samples: 120


In [2]:
import gc
import json
import torch
import torch.nn.functional as F
from transformers import TrainingArguments, Trainer

for block in model.transformer.h[:4]:
    for p in block.parameters():
        p.requires_grad = False

def collate_clauses(features):
    max_len = max(len(f["input_ids"]) for f in features)
    pad_id = tok.pad_token_id
    input_ids, attn_mask, targets = [], [], []
    for f in features:
        pad_len = max_len - len(f["input_ids"])
        input_ids.append(f["input_ids"] + [pad_id] * pad_len)
        attn_mask.append(f["attention_mask"] + [0] * pad_len)
        targets.append(f["label_token_id"])
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attn_mask, dtype=torch.long),
        "labels": torch.tensor(targets, dtype=torch.long),
    }

class ClauseTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        last_pos = inputs["attention_mask"].sum(dim=1) - 1
        hidden = model.transformer(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
        ).last_hidden_state
        pred_logits = model.lm_head(hidden[torch.arange(len(labels)), last_pos])
        loss = F.cross_entropy(pred_logits, labels)
        return (loss, pred_logits) if return_outputs else loss

training_args = TrainingArguments(
    output_dir="legalgpt-checkpoints",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=8e-4,
    optim="adafactor",
    warmup_steps=6,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    dataloader_num_workers=0,
    remove_unused_columns=False,
)

trainer = ClauseTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    data_collator=collate_clauses,
)

trainer.train()

save_dir = "legalgpt-ft"
model.save_pretrained(save_dir)
tok.save_pretrained(save_dir)
with open(f"{save_dir}/label_meta.json", "w", encoding="utf-8") as f:
    json.dump({"top_ids": TOP, "names": names}, f)

del trainer, model
gc.collect()
print(f"Saved fine-tuned model to {save_dir}/")

/opt/conda/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,48.731253
20,10.170664
30,8.554003
40,8.049201
50,7.010173
60,6.751637


Writing model shards: 100%|██████████| 1/1 [00:13<00:00, 13.64s/it]


Saved fine-tuned model to legalgpt-ft/


In [3]:
import gc
import json
import torch
from collections import Counter
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.set_num_threads(2)

with open("legalgpt-ft/label_meta.json", "r", encoding="utf-8") as f:
    meta = json.load(f)
TOP = meta["top_ids"]
names = meta["names"]

eval_tok = AutoTokenizer.from_pretrained("legalgpt-ft")
eval_model = AutoModelForCausalLM.from_pretrained("legalgpt-ft", low_cpu_mem_usage=True).eval()
label_id_to_name = {eval_tok.convert_tokens_to_ids(f"<{names[i]}>"): names[i] for i in TOP}
label_ids = list(label_id_to_name.keys())

correct = 0
confusions = Counter()

with torch.no_grad():
    for row in val_raw:
        clause = " ".join(row["text"].strip().split())[:220] + " "
        true_name = names[row["label"]]
        enc = eval_tok(clause, truncation=True, max_length=48, return_tensors="pt")
        last_pos = enc["attention_mask"].sum(dim=1) - 1
        hidden = eval_model.transformer(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"]).last_hidden_state
        logits = eval_model.lm_head(hidden[0, last_pos[0]])
        best_id = max(label_ids, key=lambda t: logits[t].item())
        pred_name = label_id_to_name[best_id]
        if pred_name == true_name:
            correct += 1
        else:
            confusions[(true_name, pred_name)] += 1

del eval_model
gc.collect()

acc = correct / len(val_raw)
print(f"Held-out accuracy ({len(val_raw)} clauses, {len(TOP)} classes): {acc:.1%} ({correct}/{len(val_raw)})\n")
print("Top confusions (True -> Predicted):")
for (t_lbl, p_lbl), count in confusions.most_common(5):
    print(f"  {t_lbl:20s} -> {p_lbl:20s} ({count})")

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 4855.33it/s]


Held-out accuracy (120 clauses, 8 classes): 48.3% (58/120)

Top confusions (True -> Predicted):
  Entire Agreements    -> Governing Laws       (11)
  Amendments           -> Governing Laws       (6)
  Severability         -> Notices              (6)
  Severability         -> Counterparts         (6)
  Amendments           -> Counterparts         (5)


In [4]:
import json
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

class LegalClauseTagger:
    def __init__(self, path="legalgpt-ft"):
        with open(f"{path}/label_meta.json", "r", encoding="utf-8") as f:
            meta = json.load(f)
        self.tok = AutoTokenizer.from_pretrained(path)
        self.model = AutoModelForCausalLM.from_pretrained(path, low_cpu_mem_usage=True).eval()
        self.label_id_to_name = {
            self.tok.convert_tokens_to_ids(f"<{meta['names'][i]}>"): meta["names"][i]
            for i in meta["top_ids"]
        }
        self.label_ids = list(self.label_id_to_name.keys())

    @torch.no_grad()
    def classify_clause(self, clause):
        prompt = " ".join(clause.strip().split())[:220] + " "
        enc = self.tok(prompt, truncation=True, max_length=48, return_tensors="pt")
        last_pos = enc["attention_mask"].sum(dim=1) - 1
        hidden = self.model.transformer(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"]).last_hidden_state
        logits = self.model.lm_head(hidden[0, last_pos[0]])
        label_logits = torch.tensor([logits[t].item() for t in self.label_ids])
        probs = F.softmax(label_logits, dim=0)
        best_idx = torch.argmax(probs).item()
        return self.label_id_to_name[self.label_ids[best_idx]], probs[best_idx].item()

tagger = LegalClauseTagger()
sample_clauses = [
    "This Agreement shall be governed by and construed in accordance with the laws of the State of New York.",
    "All notices under this Agreement shall be in writing and sent by certified mail to the addresses below.",
    "This Agreement may be executed in counterparts, each of which shall be deemed an original.",
    "This Agreement constitutes the entire agreement between the parties and supersedes all prior understandings.",
]

for clause in sample_clauses:
    label, conf = tagger.classify_clause(clause)
    print(f"{label:20s} ({conf:5.1%}) :: {clause[:58]}...")

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 95.98it/s] 


Governing Laws       (68.8%) :: This Agreement shall be governed by and construed in accor...
Notices              (19.1%) :: All notices under this Agreement shall be in writing and s...
Counterparts         (73.5%) :: This Agreement may be executed in counterparts, each of wh...
Governing Laws       (28.5%) :: This Agreement constitutes the entire agreement between th...


## Findings & Takeaways

1. **Setup & Customizations:** Expanded the task from 6 to **8 LEDGAR clause types** (`Governing Laws`, `Counterparts`, `Notices`, `Entire Agreements`, `Severability`, `Survival`, `Amendments`, and `Assignments`) using `distilgpt2` fine-tuned via a custom last-token `Trainer` (`480` training examples, `120` held-out validation clauses, `2` epochs with `adafactor`). Added softmax confidence scoring over the 8 custom label tokens inside `LegalClauseTagger.classify_clause()`.
2. **Training Progression:** Cross-entropy loss fell steadily across all 60 optimization steps (`48.73` at Step 10 $\rightarrow$ `10.17` at Step 20 $\rightarrow$ `6.75` at Step 60) as the newly added label token embeddings and upper transformer blocks adapted to legal boilerplate structure.
3. **Held-Out Evaluation & Confusions:**
   - **Validation Accuracy:** `48.3%` (`58/120` across 8 classes vs. `12.5%` random baseline).
   - **Primary Confusions:** `Entire Agreements -> Governing Laws` (`11`), `Amendments -> Governing Laws` (`6`), `Severability -> Notices` (`6`), `Severability -> Counterparts` (`6`), and `Amendments -> Counterparts` (`5`). Because all of these clauses open with nearly identical boilerplate phrasing (*"This Agreement..."*), truncating inputs to the first 48 tokens (~220 characters) causes the model to bias toward the highest-frequency class (`Governing Laws`) whenever the distinguishing legal terms appear later in the sentence.
4. **Inference & Confidence Calibration:** `LegalClauseTagger` correctly classified `Governing Laws` (`68.8%`), `Notices` (`19.1%`), and `Counterparts` (`73.5%`), while misclassifying the `Entire Agreements` test prompt as `Governing Laws` with a noticeably lower confidence (`28.5%`)—mirroring the exact confusion pattern seen on the validation set.